# SemEval 2026 EmoVA — Final Evaluation (Task 2a: Forecasting)

**Flow:**
1. Train on the **full** Task 2a training set for exactly `N` epochs (from `config.json`)
2. Run inference on the forecasting test users
3. Evaluate against official gold labels (`test_labels_subtask2a_and_2b.csv`)

**Note on test evaluation:**  
The gold labels contain **one aggregate state-change value per user** (not per window).  
We therefore take the **last window prediction** for each user as the final forecast and compute
**between-user Pearson r** for valence and arousal separately.  
The overall score is their average — consistent with the Task 2a ranking metric.

> Set `CONFIG_PATH` in *Cell 5* to point to the `config.json` produced by your best 2a ablation run.

In [ ]:
import os

def setup_storage():
    # Google Colab
    if "COLAB_GPU" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        base_dir = "/content/drive/MyDrive"
        env = "colab"
    # Kaggle
    elif os.path.exists("/kaggle"):
        base_dir = "/kaggle/working"
        env = "kaggle"
    # Local fallback
    else:
        base_dir = os.getcwd()
        env = "local"

    print(f"Running on : {env}")
    print(f"Base dir   : {base_dir}")
    return base_dir, env


BASE_DIR, ENV = setup_storage()

In [ ]:
PROJECT_ROOT = f"{BASE_DIR}/SEMEVAL2026_EMOVA"
CKPT_DIR     = f"{PROJECT_ROOT}/model_checkpoints_final_2a"

os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
!git clone https://github.com/AndreaLolli2912/SemEval2026-EmoVA.git
%cd SemEval2026-EmoVA

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr as scipy_pearsonr
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

from src.data.dataset import EmoVADataset2a
from src.data.collate import create_collate_fn_2a
from src.models.affect_model import AffectModel2a
from src.models.tokenizer_wrapper import TokenizerWrapper
from src.training.trainer_2a import train_epoch as train_epoch_2a
from src.training import GradientClipper

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    torch.use_deterministic_algorithms(True)

## Config

Point `CONFIG_PATH` to the `config.json` saved by your best 2a ablation run.  
All string-encoded values (`"True"`, `"32"`, …) are automatically converted back to proper Python types.

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
CONFIG_PATH = f"{PROJECT_ROOT}/model_checkpoints_task2/<YOUR_RUN_FOLDER>/config.json"
# ──────────────────────────────────────────────────────────────────────────────


def _parse(v):
    """Convert string-encoded values produced by the trainer back to Python types."""
    if not isinstance(v, str):
        return v
    if v == 'True':  return True
    if v == 'False': return False
    if v == 'None':  return None
    try:    return int(v)
    except ValueError: pass
    try:    return float(v)
    except ValueError: pass
    return v


with open(CONFIG_PATH) as f:
    raw = json.load(f)

cfg_dict = {k: _parse(v) for k, v in raw.items()}
config   = type('Config', (), cfg_dict)()

# ── Derived paths (override data_path to local project structure) ─────────────
config.data_path = f"{PROJECT_ROOT}/dataset/train_subtask2a.csv"

# ── 2a-specific hyperparams (match the ablation settings) ─────────────────────
MAX_HISTORY = getattr(config, 'max_history', 5)   # history window length
STEP        = getattr(config, 'step', 2)           # sliding-window stride

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(config.seed)

print(f"Model       : {config.model_name}")
print(f"Epochs      : {config.epochs}")
print(f"Max history : {MAX_HISTORY}")
print(f"Step        : {STEP}")
print(f"LoRA        : {getattr(config, 'encoder_lora', False)}")
print(f"BitFit      : {getattr(config, 'encoder_bitfit', False)}")
print(f"Device      : {device}")

## Data — Full Task 2a Training Set (no validation split)

In [ ]:
tokenizer    = TokenizerWrapper(config.model_name, config.max_text_length)
full_dataset = EmoVADataset2a(
    path=config.data_path,
    dtype=torch.float32,
    constrain_output=config.constrain_output,
    max_history=MAX_HISTORY,
    step=STEP,
)
collate_fn = create_collate_fn_2a(tokenizer)

train_loader = DataLoader(
    full_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=config.num_workers,
)

print(f"Full training dataset: {len(full_dataset)} windows (across all users)")

## Model

In [ ]:
model = AffectModel2a(
    model_path=config.model_name,
    encoder_bitfit=getattr(config, 'encoder_bitfit', False),
    encoder_use_lora=getattr(config, 'encoder_lora', False),
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    n_heads=config.n_heads,
    isab_inducing_points=config.isab_inducing_points,
    pma_num_seeds=config.pma_num_seeds,
    lstm_hidden_dim=config.lstm_hidden_dim,
    lstm_num_layers=config.lstm_num_layers,
    lstm_bidirectional=config.lstm_bidirectional,
    dropout=config.dropout,
    constrain_output=config.constrain_output,
)

if getattr(config, 'encoder_bitfit', False) or getattr(config, 'encoder_lora', False):
    model.encoder.backbone.gradient_checkpointing_enable()

model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total, {trainable:,} trainable")

## Optimizer / Scheduler / Clipper

In [ ]:
param_groups = [
    {'params': [p for n, p in model.encoder.named_parameters() if p.requires_grad],
     'lr': 5e-6, 'name': 'encoder'},
    ({'params': list(model.isab.parameters()), 'lr': config.lr, 'name': 'isab'}
     if model.isab else None),
    ({'params': list(model.pma.parameters()), 'lr': config.lr, 'name': 'pma'}
     if model.pma else None),
    {'params': list(model.lstm.parameters()), 'lr': config.lr, 'name': 'lstm'},
    {'params': list(model.head.parameters()), 'lr': config.lr, 'name': 'head'},
]
param_groups = [pg for pg in param_groups if pg is not None and len(pg['params']) > 0]

optimizer = AdamW(param_groups, weight_decay=config.weight_decay)
# Step on train loss (no val set available during final training)
scheduler = ReduceLROnPlateau(
    optimizer, mode='min',
    factor=config.scheduler_factor,
    patience=config.scheduler_patience,
)
clipper = GradientClipper(max_norm=config.max_grad_norm)

for pg in optimizer.param_groups:
    n = sum(p.numel() for p in pg['params'])
    print(f"{pg['name']:12s}: {n:>12,} params  lr={pg['lr']:.1e}")

## Training — Full Dataset

Train for exactly `config.epochs` epochs (the best epoch found in ablation).  
Note: `trainer_2a.train_epoch` uses MSE loss internally (no `loss_fn_name` argument).

In [ ]:
n_epochs = config.epochs
history  = {'train_loss': []}

print(f"Training for {n_epochs} epochs on {len(full_dataset)} windows")
print("=" * 60)

for epoch in range(n_epochs):
    # trainer_2a.train_epoch does NOT take a loss_fn_name argument (uses MSE internally)
    result = train_epoch_2a(
        model, train_loader,
        optimizer, device, config,
        clipper=clipper,
    )

    train_loss = result['loss']
    current_lr = optimizer.param_groups[-1]['lr']

    scheduler.step(train_loss)

    history['train_loss'].append(train_loss)

    print(
        f"Epoch {epoch+1:3d}/{n_epochs} "
        f"| MSE Loss: {train_loss:.4f} "
        f"| LR: {current_lr:.2e}"
    )

print("\nTraining complete!")

## Save Final Checkpoint

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir   = Path(CKPT_DIR) / f"{timestamp}_final_eval_2a"
run_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'config': cfg_dict,
        'history': history,
    },
    run_dir / 'final_checkpoint.pt',
)

print(f"Checkpoint saved to: {run_dir}")

## Test Inference

**Data sources:**
- `subtask2a_forecasting_user_marker.csv` — history texts for all users (from the test release),
  marked with `is_forecasting_user` to identify who needs a forecast
- `test_labels_subtask2a_and_2b.csv` — one gold state-change value per forecasting user

**Strategy:** For each forecasting user we reconstruct all sliding windows from their history
using the same `max_history` / `step` as training, run the model on every window, and keep
the **last window's prediction** (most context available) as the per-user forecast.
Predictions are stored in `user_last_pred[user_id] -> np.array([pred_valence, pred_arousal])`.

In [ ]:
MARKER_PATH = (
    f"{PROJECT_ROOT}/dataset/TEST_RELEASE_5JAN2026/subtask2a_forecasting_user_marker.csv"
)
GOLD_PATH = (
    f"{PROJECT_ROOT}/dataset/TEST_LABELS_RELEASE_23FEB2026/test_labels_subtask2a_and_2b.csv"
)

# Load gold labels (one row per forecasting user)
gold_df = pd.read_csv(GOLD_PATH).set_index('user_id')
forecasting_user_ids = set(gold_df.index.tolist())
print(f"Forecasting users (gold): {len(forecasting_user_ids)}")
print(gold_df.head())

In [ ]:
# Build EmoVADataset2a from the marker file, then collect predictions per user
test_dataset_2a = EmoVADataset2a(
    path=MARKER_PATH,
    dtype=torch.float32,
    constrain_output=config.constrain_output,
    max_history=MAX_HISTORY,
    step=STEP,
)

test_loader_2a = DataLoader(
    test_dataset_2a,
    batch_size=config.batch_size,
    shuffle=False,          # keep chronological order so last == most recent
    collate_fn=collate_fn,
    num_workers=0,
)

print(f"Test dataset: {len(test_dataset_2a)} windows")

In [ ]:
# Collect ALL per-window predictions, grouped by user
model.eval()
user_all_preds = {}   # user_id -> list of [2] arrays (one per window)

with torch.no_grad():
    for batch in test_loader_2a:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        history_va     = batch['history_va'].to(device)
        seq_lengths    = batch['seq_lengths'].to(device)
        seq_mask       = batch['seq_attention_mask'].to(device)
        user_ids       = batch['user_ids']

        preds = model(
            input_ids, attention_mask, history_va, seq_lengths, seq_mask
        )  # [B, 2]
        preds_np = preds.cpu().numpy()  # [B, 2]

        for i, uid in enumerate(user_ids):
            if uid not in user_all_preds:
                user_all_preds[uid] = []
            user_all_preds[uid].append(preds_np[i])   # [2]

print(f"Users with predictions: {len(user_all_preds)}")

# Take the LAST window prediction for each forecasting user
user_last_pred = {
    uid: np.array(preds_list[-1])           # most-recent window
    for uid, preds_list in user_all_preds.items()
    if uid in forecasting_user_ids
}

print(f"Forecasting users with predictions: {len(user_last_pred)}")
missing = forecasting_user_ids - set(user_last_pred.keys())
if missing:
    print(f"  WARNING: no prediction for {len(missing)} users: {missing}")

## Official Evaluation

The gold labels are **one value per user**, so we compute **between-user Pearson r**
(correlation of predicted state-change vs actual state-change across users).

The overall score is the average of valence r and arousal r — the Task 2a ranking metric.

In [ ]:
# Align predictions and gold for matched users only
eval_user_ids = sorted(user_last_pred.keys())

pred_v = np.array([user_last_pred[uid][0] for uid in eval_user_ids])
pred_a = np.array([user_last_pred[uid][1] for uid in eval_user_ids])
gold_v = np.array([gold_df.loc[uid, 'state_change_valence'] for uid in eval_user_ids])
gold_a = np.array([gold_df.loc[uid, 'state_change_arousal'] for uid in eval_user_ids])

r_valence, _ = scipy_pearsonr(pred_v, gold_v)
r_arousal, _ = scipy_pearsonr(pred_a, gold_a)
mae_valence = float(np.mean(np.abs(pred_v - gold_v)))
mae_arousal = float(np.mean(np.abs(pred_a - gold_a)))
overall_score = (r_valence + r_arousal) / 2.0

print("\n" + "=" * 60)
print("SemEval 2026 EmoVA — Task 2a: Test Set Evaluation")
print("=" * 60)
print(f"\nUsers evaluated : {len(eval_user_ids)}")
print("\nVALENCE")
print(f"  Pearson r (between-user): {r_valence:>7.4f}")
print(f"  MAE                     : {mae_valence:>7.4f}")
print("\nAROUSAL")
print(f"  Pearson r (between-user): {r_arousal:>7.4f}")
print(f"  MAE                     : {mae_arousal:>7.4f}")
print("\n" + "-" * 60)
print(f"OVERALL SCORE (mean r)    : {overall_score:>7.4f}  <- ranking metric")
print("=" * 60 + "\n")

## Summary Table & Scatter Plots

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame([
    {'Dimension': 'Valence', 'Between-user r': r_valence, 'MAE': mae_valence,
     'Pred mean': pred_v.mean(), 'Gold mean': gold_v.mean()},
    {'Dimension': 'Arousal', 'Between-user r': r_arousal, 'MAE': mae_arousal,
     'Pred mean': pred_a.mean(), 'Gold mean': gold_a.mean()},
]).set_index('Dimension').round(4)

print(summary.to_string())
print(f"\nOVERALL SCORE (ranking metric): {overall_score:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, gm, pm, dim, color, r in [
    (axes[0], gold_v, pred_v, 'Valence', 'steelblue', r_valence),
    (axes[1], gold_a, pred_a, 'Arousal', 'seagreen',  r_arousal),
]:
    ax.scatter(gm, pm, alpha=0.7, s=40, c=color)
    lims = [min(gm.min(), pm.min()) - 0.2, max(gm.max(), pm.max()) + 0.2]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    ax.set_xlabel(f'Gold {dim} (state change)')
    ax.set_ylabel(f'Predicted {dim} (state change)')
    ax.set_title(f"{dim} — r = {r:.4f}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Task 2a — Overall Score = {overall_score:.4f}",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(str(run_dir / 'test_scatter_2a.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {run_dir / 'test_scatter_2a.png'}")

## Save Evaluation Results

In [ ]:
eval_results = {
    'valence/r_between': float(r_valence),
    'valence/mae':       float(mae_valence),
    'arousal/r_between': float(r_arousal),
    'arousal/mae':       float(mae_arousal),
    'overall/score':     float(overall_score),
    'n_users':           len(eval_user_ids),
}

results_path = run_dir / 'eval_results_2a.json'
with open(results_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"Results saved to: {results_path}")